In [247]:
import numpy as np
import pandas as pd
import random

In [248]:
pd.set_option('display.max_columns', None)

In [249]:
dataset_workout = pd.read_csv('../../datas/dataset_workout.csv')

In [250]:
df = pd.read_csv('../../datas/dataset_users.csv')
df_weekly_progress = pd.read_csv('../../datas/dataset_userprogress.csv')

df_merged = pd.merge(
    df,                       # Tabel kiri (Data Profil)
    df_weekly_progress,       # Tabel kanan (Data Mingguan)
    on='User_ID',             # Kunci Penghubung
    how='left'                # Jenis Join
)

In [251]:
# # --- 1. DEFINISI OTOT YANG LEBIH SPESIFIK (SUB-GROUPS) ---
# MUSCLE_GROUPS = {
#     # Upper Body - Push
#     'Chest': ['pectorals', 'chest', 'serratus anterior'],
#     'Shoulders': ['delts', 'deltoids', 'shoulders', 'rotator cuff'],
#     'Triceps': ['triceps'],
    
#     # Upper Body - Pull
#     'Back': ['lats', 'latissimus dorsi', 'upper back', 'lower back', 'spine'],
#     'Traps': ['trapezius', 'traps', 'neck'],
#     'Biceps': ['biceps', 'brachialis', 'forearms'],
    
#     # Lower Body (Dipecah biar variatif)
#     'Quads': ['quads', 'quadriceps', 'thigh', 'legs'],     # Fokus paha depan
#     'Hamstrings': ['hamstrings', 'legs', 'back of leg'],   # Fokus paha belakang
#     'Glutes': ['glutes', 'butt', 'hips', 'abductors'],     # Fokus bokong
#     'Calves': ['calves', 'calfs', 'lower leg'],            # Fokus betis
#     'Legs_General': ['legs', 'thigh', 'quads', 'hamstrings'], # Fallback jika butuh umum
    
#     # Core & Cardio
#     'Abs': ['abs', 'abdominals', 'obliques', 'core', 'waist'],
#     'Cardio': ['cardio', 'cardiovascular']
# }

# SPORTS_COLUMNS = [
#     'Badminton_x', 'Football_x', 'Basketball_x', 'Tennis_x', 
#     'Volleyball_x', 'Table_Tennis_x', 'Swim_x'
# ]

In [252]:
MUSCLE_GROUPS = {
    # Spesifik (Untuk variasi)
    'Chest': ['pectorals', 'chest', 'serratus anterior'],
    'Shoulders': ['delts', 'deltoids', 'shoulders', 'rotator cuff'],
    'Triceps': ['triceps'],
    'Back': ['lats', 'latissimus dorsi', 'upper back', 'lower back', 'spine'],
    'Traps': ['trapezius', 'traps', 'neck'],
    'Biceps': ['biceps', 'brachialis', 'forearms'],
    'Quads': ['quads', 'quadriceps', 'thigh'], 
    'Hamstrings': ['hamstrings', 'legs', 'back of leg'], 
    'Glutes': ['glutes', 'butt', 'hips', 'abductors'], 
    'Calves': ['calves', 'calfs', 'lower leg'],
    'Abs': ['abs', 'abdominals', 'obliques', 'core', 'waist'],
    'Cardio': ['cardio', 'cardiovascular'],

    # Kategori Umum (FALLBACK - Penyelamat jika data spesifik kosong)
    'Push_General': ['chest', 'shoulders', 'triceps', 'push'],
    'Pull_General': ['back', 'biceps', 'pull'],
    'Legs_General': ['legs', 'thigh', 'quads', 'hamstrings', 'glutes', 'lower body']
}

SPORTS_COLUMNS = [
    'Badminton_x', 'Football_x', 'Basketball_x', 'Tennis_x', 
    'Volleyball_x', 'Table_Tennis_x', 'Swim_x'
]

In [253]:
def clean_string_data(text):
    """
    Fungsi untuk mengubah "['cable']" menjadi "cable"
    Sangat berguna karena CSV membaca list sebagai teks biasa.
    """
    if pd.isna(text): 
        return "None"
    # Menghapus kurung siku dan tanda kutip satu per satu
    cleaned = str(text).replace("[", "").replace("]", "").replace("'", "")
    return cleaned.strip()

In [254]:
def get_exercises_by_muscle(df_exercises, target_groups, target_env, limit=3):
    # 1. Filter Environment: 
    # Hanya ambil data yang environment-nya SAMA dengan user, ATAU 'Other' (untuk olahraga umum)
    # Tapi untuk latihan beban spesifik, kita strict ke target_env
    df_env = df_exercises[df_exercises['Environment'] == target_env]
    
    keywords = []
    for group in target_groups:
        keywords.extend(MUSCLE_GROUPS.get(group, []))
    
    # 2. Cari yang ototnya cocok
    mask = df_env['targetMuscles'].astype(str).apply(
        lambda x: any(k.lower() in x.lower() for k in keywords)
    )
    
    df_filtered = df_env[mask]
    
    if len(df_filtered) > 0:
        return df_filtered.sample(n=min(limit, len(df_filtered)), replace=False)
    else:
        return pd.DataFrame()

In [255]:
def generate_weekly_plan(user_row, df_exercises):
    user_id = user_row['User_ID']
    freq = user_row['Workout_Frequency_x'] 
    duration = user_row['Average_Duration_Minutes_x']
    goal = user_row['Goal_x']
    
    # --- 1. TENTUKAN ENVIRONMENT UTAMA (Random Gym/Home) ---
    main_env = random.choice(['Gym', 'Home'])
    
    # --- 2. CEK HOBI OLAHRAGA ---
    user_sports = []
    for col in SPORTS_COLUMNS:
        if user_row.get(col, 0) == 1:
            sport_name = col.replace('_x', '').replace('_', ' ')
            user_sports.append(sport_name)
    
    # --- 3. HITUNG TARGET JUMLAH LATIHAN ---
    target_exercises = max(5, int(duration / 7))
    target_exercises = min(target_exercises, 12) 
    
    weekly_schedule = []
    
    # --- 4. PEMBAGIAN HARI (SPLIT) ---
    schedule_map = {}
    
    if freq <= 2:
        schedule_map = {
            1: {'Theme': 'Full Body A', 'Focus': ['Quads', 'Chest', 'Shoulders', 'Triceps', 'Abs']},
            2: {'Theme': 'Full Body B', 'Focus': ['Hamstrings', 'Glutes', 'Back', 'Biceps', 'Calves']}
        }
    elif freq == 3:
        schedule_map = {
            1: {'Theme': 'Push Day', 'Focus': ['Chest', 'Shoulders', 'Triceps', 'Abs']},
            2: {'Theme': 'Pull Day', 'Focus': ['Back', 'Traps', 'Biceps', 'Abs']},
            3: {'Theme': 'Leg Day', 'Focus': ['Quads', 'Hamstrings', 'Glutes', 'Calves']}
        }
    elif freq == 4:
        schedule_map = {
            1: {'Theme': 'Upper Power', 'Focus': ['Chest', 'Back', 'Shoulders']},
            2: {'Theme': 'Lower Power', 'Focus': ['Quads', 'Hamstrings', 'Calves']},
            3: {'Theme': 'Upper Hypertrophy', 'Focus': ['Biceps', 'Triceps', 'Chest', 'Back']},
            4: {'Theme': 'Lower Hypertrophy', 'Focus': ['Glutes', 'Quads', 'Abs']}
        }
    else: 
        schedule_map = {
            1: {'Theme': 'Chest & Tris', 'Focus': ['Chest', 'Triceps', 'Abs']},
            2: {'Theme': 'Back & Bis', 'Focus': ['Back', 'Biceps', 'Traps']},
            3: {'Theme': 'Legs Quad Focus', 'Focus': ['Quads', 'Calves', 'Abs']},
            4: {'Theme': 'Shoulders & Abs', 'Focus': ['Shoulders', 'Abs']},
            5: {'Theme': 'Legs Glute/Ham', 'Focus': ['Hamstrings', 'Glutes']},
            6: {'Theme': 'Cardio & Core', 'Focus': ['Abs', 'Cardio']}
        }

    # --- 5. LOOPING JADWAL ---
    for day_num, config in schedule_map.items():
        if day_num > freq: break
            
        theme = config['Theme']
        day_muscles = list(config['Focus']) 
        
        # --- LOGIKA BARU: INJEKSI SPORT "OTHER" UNTUK SEMUA USER ---
        # Tentukan apakah hari ini user dapet slot 'Cardio' (yang isinya bisa Sport Other)?
        add_cardio = False

        # 1. PRIORITAS HOBI (Berlaku untuk SEMUA GOAL termasuk Muscle Gain)
        # Jika user punya hobi olahraga, kasih chance 70% muncul di hari apapun
        if len(user_sports) > 0:
            if random.random() < 0.7: 
                add_cardio = True
        
        # 2. Logic Goal (Fallback jika tidak dapet slot dari hobi)
        elif goal == 'Weight Loss':
            add_cardio = True # Wajib ada
        elif goal == 'Maintain':
            if random.random() > 0.5: # 50% chance
                add_cardio = True
        else:
            # Muscle Gain tanpa hobi: Masih dikasih chance kecil (20%) biar variatif
            if random.random() < 0.2:
                add_cardio = True

        # Eksekusi penambahan slot Cardio/Sport ke dalam list otot hari ini
        if add_cardio and 'Cardio' not in day_muscles:
            day_muscles.append('Cardio')
        
        # --- ALOKASI SLOT ---
        num_groups = len(day_muscles)
        base_slot = target_exercises // num_groups
        remainder = target_exercises % num_groups
        
        for i, muscle in enumerate(day_muscles):
            my_limit = base_slot
            if i < remainder: 
                my_limit += 1
            
            my_limit = max(1, my_limit)
            
            # === LOGIKA CARDIO / SPORT (OTHER) ===
            if muscle == 'Cardio':
                # Cek olahraga external (User Sports)
                # Ini akan ketrigger sering karena logika "Prioritas Hobi" diatas
                if len(user_sports) > 0:
                    sport = random.choice(user_sports)
                    weekly_schedule.append({
                        'User_ID': user_id,
                        'Day': f"Day {day_num} - {theme}",
                        'Muscle Group': 'Cardio',
                        'Exercise Name': sport,
                        'Equipment': 'Sport Equipment',
                        'Sets': 1,
                        'Reps': f"{random.randint(20, 45)} Mins",
                        'Instructions': f"Play {sport} for endurance and fun.",
                        'Environment': 'Other' # <-- Variasi Other muncul disini
                    })
                else:
                    # Cardio DB (Sesuai Main Env)
                    # Hanya kepanggil kalau user ga punya hobi olahraga di data profilnya
                    cardio_ex = get_exercises_by_muscle(df_exercises, ['Cardio'], main_env, limit=1)
                    for _, ex in cardio_ex.iterrows():
                        weekly_schedule.append({
                            'User_ID': user_id,
                            'Day': f"Day {day_num} - {theme}",
                            'Muscle Group': 'Cardio',
                            'Exercise Name': ex['name'],
                            'Equipment': clean_string_data(ex['equipments']),
                            'Sets': 1,
                            'Reps': f"{random.randint(15, 30)} Mins",
                            'Instructions': clean_string_data(ex['instructions']),
                            'Environment': ex['Environment']
                        })
            
            # === LOGIKA OTOT BIASA ===
            else:
                exercises = get_exercises_by_muscle(df_exercises, [muscle], main_env, limit=my_limit)
                
                # Fallback Logic (Sesuai request sebelumnya biar ga kosong)
                if exercises.empty and muscle in ['Quads', 'Hamstrings', 'Glutes', 'Calves']:
                    exercises = get_exercises_by_muscle(df_exercises, ['Legs_General'], main_env, limit=my_limit)
                
                for _, ex in exercises.iterrows():
                    # Sets & Reps
                    if goal == 'Muscle Gain':
                        sets, reps = 3, "8-12"
                    elif goal == 'Weight Loss':
                        sets, reps = 4, "12-15"
                    else:
                        sets, reps = 3, "10-12"
                    
                    weekly_schedule.append({
                        'User_ID': user_id,
                        'Day': f"Day {day_num} - {theme}",
                        'Muscle Group': muscle,
                        'Exercise Name': ex['name'],
                        'Equipment': clean_string_data(ex['equipments']),
                        'Sets': sets,
                        'Reps': reps,
                        'Instructions': clean_string_data(ex['instructions']),
                        'Environment': ex['Environment']
                    })

    return pd.DataFrame(weekly_schedule)

In [256]:
all_user_plans = []
unique_users_df = df_merged.drop_duplicates(subset=['User_ID'])

print("Memulai generate jadwal (Environment akan diacak per user)...")

for index, user_row in unique_users_df.iterrows():
    my_plan = generate_weekly_plan(user_row, dataset_workout)
    
    if not my_plan.empty:
        all_user_plans.append(my_plan)

# Gabung semua
if len(all_user_plans) > 0:
    df_all_plans = pd.concat(all_user_plans, ignore_index=True)
    print(f"Sukses! Dibuat {len(df_all_plans)} baris jadwal.")
else:
    print("Warning: Tidak ada jadwal terbentuk.")

Memulai generate jadwal (Environment akan diacak per user)...
Sukses! Dibuat 1957 baris jadwal.


In [257]:
df_final = pd.merge(
    df_merged,      # Data Mingguan (Week 0-12)
    df_all_plans,   # Data Latihan (Day 1-X)
    on='User_ID',   # Disambung pake User ID
    how='left'      # Left Join
)

print(f"Sukses! df_final berhasil dibuat dengan ukuran: {df_final.shape}")

Sukses! df_final berhasil dibuat dengan ukuran: (25441, 62)


In [258]:
df_final.head()

,User_ID,Age_x,Gender_x,Height_cm_x,Initial_Weight_kg_x,Initial_BMI_x,BMI_Category,Body_Fat_Category_x,Body_Fat_Percentage,Goal_x,Workout_Frequency_x,Average_Duration_Minutes_x,level_x,Badminton_x,Football_x,Basketball_x,Tennis_x,Volleyball_x,Table_Tennis_x,Swim_x,Age_y,Gender_y,Height_cm_y,Initial_Weight_kg_y,Initial_BMI_y,BMI_Category_x,Body_Fat_Category_y,Body_Fat_Percentage_x,Goal_y,Workout_Frequency_y,Average_Duration_Minutes_y,level_y,Badminton_y,Football_y,Basketball_y,Tennis_y,Volleyball_y,Table_Tennis_y,Swim_y,Week,Weight_kg,BMI,Body_Fat_Percentage_y,Daily_Calories,Daily_Water_ml,Target_Protein_g,Target_Carbs_g,Target_Fat_g,Limit_Sugar_g,Target_Fiber_g,Limit_Cholesterol_mg,Target_Calcium_mg,Meal_Frequency,BMI_Category_y,Day,Muscle Group,Exercise Name,Equipment,Sets,Reps,Instructions,Environment
0,1,45,Male,181,53,16.18,Underweight,2.0,10.9,Muscle Gain,2,90,Beginner,0,1,0,0,1,1,0,45,Male,181,53,16.18,Underweight,2.0,10.9,Muscle Gain,2,90,Beginner,0,1,0,0,1,1,0,0,53.0,16.18,10.9,2281,2755,171,285,50,57,31,300,1000,5,Underweight,Day 1 - Full Body A,Quads,bodyweight squat,body weight,3,8-12,"Step:1 Stand with feet shoulder-width apart., ...",Home
1,1,45,Male,181,53,16.18,Underweight,2.0,10.9,Muscle Gain,2,90,Beginner,0,1,0,0,1,1,0,45,Male,181,53,16.18,Underweight,2.0,10.9,Muscle Gain,2,90,Beginner,0,1,0,0,1,1,0,0,53.0,16.18,10.9,2281,2755,171,285,50,57,31,300,1000,5,Underweight,Day 1 - Full Body A,Quads,reverse lunge,body weight,3,8-12,"Step:1 Stand tall., Step:2 Step one foot back ...",Home
2,1,45,Male,181,53,16.18,Underweight,2.0,10.9,Muscle Gain,2,90,Beginner,0,1,0,0,1,1,0,45,Male,181,53,16.18,Underweight,2.0,10.9,Muscle Gain,2,90,Beginner,0,1,0,0,1,1,0,0,53.0,16.18,10.9,2281,2755,171,285,50,57,31,300,1000,5,Underweight,Day 1 - Full Body A,Shoulders,pike push up,body weight,3,8-12,"Step:1 Start in a plank position, then lift yo...",Home
3,1,45,Male,181,53,16.18,Underweight,2.0,10.9,Muscle Gain,2,90,Beginner,0,1,0,0,1,1,0,45,Male,181,53,16.18,Underweight,2.0,10.9,Muscle Gain,2,90,Beginner,0,1,0,0,1,1,0,0,53.0,16.18,10.9,2281,2755,171,285,50,57,31,300,1000,5,Underweight,Day 1 - Full Body A,Triceps,impossible dips,body weight,3,8-12,Step:1 Position yourself between two parallel ...,Home
4,1,45,Male,181,53,16.18,Underweight,2.0,10.9,Muscle Gain,2,90,Beginner,0,1,0,0,1,1,0,45,Male,181,53,16.18,Underweight,2.0,10.9,Muscle Gain,2,90,Beginner,0,1,0,0,1,1,0,0,53.0,16.18,10.9,2281,2755,171,285,50,57,31,300,1000,5,Underweight,Day 1 - Full Body A,Abs,bicycle crunch,body weight,3,8-12,"Step:1 Lie on back, hands behind head., Step:2...",Home


In [259]:
# List ID yang mau dilihat
target_ids = [2]

# Filter df_final
result_df = df_final[df_final['User_ID'].isin(target_ids)]

# Urutkan biar rapi
result_df = result_df.sort_values(by=['User_ID', 'Week', 'Day'])

# Tampilkan kolom penting saja
cols_view = ['User_ID', 'Week', 'Day', 'Goal_x', 'Muscle Group', 'Exercise Name', 'Sets', 'Reps']

display(result_df[cols_view].head(20))

,User_ID,Week,Day,Goal_x,Muscle Group,Exercise Name,Sets,Reps
156,2,0,Day 1 - Push Day,Muscle Gain,Shoulders,pike push up,3,8-12
157,2,0,Day 1 - Push Day,Muscle Gain,Triceps,impossible dips,3,8-12
158,2,0,Day 1 - Push Day,Muscle Gain,Abs,bicycle crunch,3,8-12
159,2,0,Day 1 - Push Day,Muscle Gain,Cardio,Table Tennis,1,43 Mins
160,2,0,Day 2 - Pull Day,Muscle Gain,Back,superman,3,8-12
161,2,0,Day 2 - Pull Day,Muscle Gain,Abs,bicycle crunch,3,8-12
162,2,0,Day 3 - Leg Day,Muscle Gain,Quads,bodyweight squat,3,8-12
163,2,0,Day 3 - Leg Day,Muscle Gain,Quads,reverse lunge,3,8-12
164,2,0,Day 3 - Leg Day,Muscle Gain,Hamstrings,bodyweight squat,3,8-12
165,2,0,Day 3 - Leg Day,Muscle Gain,Glutes,reverse lunge,3,8-12


In [260]:
df_final.to_csv('../../datas/dataset_final.csv', index=False)